# 3. Data Augmentation & Model Training

**Methodology (important):** the real data is split into train/test **first**, and the test set
stays 100% real throughout — synthetic rows are added **only** to the training set, and the
generative model itself was fit *only* on training-split fraud rows (Notebook 2). This avoids
synthetic-data leakage into evaluation at both the classifier level and the generator level, so
the numbers below reflect a genuinely fair read on real-world fraud detection.

## 3.1 Split real data first (test set = 100% real, held out)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                              classification_report, average_precision_score, precision_recall_curve)
import joblib

RANDOM_STATE = 42
df = pd.read_csv("../Data/creditcard.csv")
synthetic_fraud = pd.read_csv("../Data/synthetic_fraud.csv")
FEATURES = [c for c in df.columns if c != "Class"]

real_train, real_test = train_test_split(df, test_size=0.3, stratify=df["Class"], random_state=RANDOM_STATE)
print(f"Real train: {real_train.shape}, fraud={real_train['Class'].sum()}")
print(f"Real test : {real_test.shape}, fraud={real_test['Class'].sum()} (held out, never touched by synthetic data)")

Real train: (199364, 31), fraud=344
Real test : (85443, 31), fraud=148 (held out, never touched by synthetic data)


## 3.2 Build the two training sets

In [1]:
X_test, y_test = real_test[FEATURES], real_test["Class"]
X_train_base, y_train_base = real_train[FEATURES], real_train["Class"]

augmented_train = pd.concat([real_train, synthetic_fraud], ignore_index=True)
augmented_train = augmented_train.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)
X_train_aug, y_train_aug = augmented_train[FEATURES], augmented_train["Class"]
augmented_train.to_csv("../Data/augmented_dataset.csv", index=False)

print(f"Baseline training fraud rows  : {y_train_base.sum()}")
print(f"Augmented training fraud rows : {y_train_aug.sum()} ({real_train['Class'].sum()} real + {synthetic_fraud['Class'].sum()} synthetic)")

Baseline training fraud rows  : 344
Augmented training fraud rows : 5344 (344 real + 5000 synthetic)


## 3.3 Train identical Random Forest classifiers on each training set
Same hyperparameters for both (`n_estimators=100, max_depth=16` — capped depth chosen for training speed in this single-core sandbox; a production run could search this properly). The *only* difference between the two models is whether synthetic fraud rows were added.

In [1]:
def make_model():
    return RandomForestClassifier(n_estimators=100, max_depth=16, random_state=RANDOM_STATE, n_jobs=-1)

clf_baseline = make_model()
clf_baseline.fit(X_train_base, y_train_base)

clf_augmented = make_model()
clf_augmented.fit(X_train_aug, y_train_aug)
print("Both models trained.")

Both models trained.


## 3.4 Evaluate both models on the SAME real-only held-out test set

In [1]:
def evaluate(clf, name):
    proba = clf.predict_proba(X_test)[:, 1]
    pred = clf.predict(X_test)
    print(f"--- {name} ---")
    print(classification_report(y_test, pred, target_names=["Non-Fraud","Fraud"], zero_division=0))
    print(f"AUC: {roc_auc_score(y_test, proba):.4f}\n")
    return pred, proba

pred_base, proba_base = evaluate(clf_baseline, "BASELINE (real data only)")
pred_aug, proba_aug = evaluate(clf_augmented, "AUGMENTED (real + synthetic fraud)")

--- BASELINE (real data only) ---
              precision    recall  f1-score   support

   Non-Fraud       1.00      1.00      1.00     85295
       Fraud       0.96      0.76      0.85       148

    accuracy                           1.00     85443
   macro avg       0.98      0.88      0.92     85443
weighted avg       1.00      1.00      1.00     85443

AUC: 0.9671

--- AUGMENTED (real + synthetic fraud) ---
              precision    recall  f1-score   support

   Non-Fraud       1.00      1.00      1.00     85295
       Fraud       0.83      0.82      0.83       148

    accuracy                           1.00     85443
   macro avg       0.91      0.91      0.91     85443
weighted avg       1.00      1.00      1.00     85443

AUC: 0.9692



## 3.5 Results at the default threshold (0.5)

| Model | Precision | Recall | F1-score | AUC |
|---|---|---|---|---|
| Without Synthetic (Baseline) | 0.957 | 0.757 | 0.845 | 0.967 |
| With Synthetic (Augmented) | 0.830 | 0.824 | 0.827 | 0.969 |

At the default threshold, augmentation trades precision for recall: **+6.8 points of recall**,
but **F1 actually dips slightly (-1.8 points)** and precision drops noticeably. 

## 3.6 Save the recommended model

In [1]:
joblib.dump(clf_augmented, "../Model/fraud_model_rf.pkl")
print("Saved -> Model/fraud_model_rf.pkl")

Saved -> Model/fraud_model_rf.pkl
